In [1]:
!pip install google-cloud-bigquery requests pandas pyarrow

In [2]:
import requests

url = "https://api.nbp.pl/api/exchangerates/rates/a/eur/?format=json"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    kurs = data['rates'][0]['mid']
    print(f" Sukces! Połączenie z NBP działa. Aktualny kurs EUR to: {kurs} zł")
else:
    print(f" Coś poszło nie tak. Kod odpowiedzi serwera: {response.status_code}")

 Sukces! Połączenie z NBP działa. Aktualny kurs EUR to: 4.301 zł


In [3]:
from google.cloud import bigquery
print(" Biblioteka Google Cloud załadowała się poprawnie!")

 Biblioteka Google Cloud załadowała się poprawnie!


In [4]:
import os
print("Twój Jupyter znajduje się w folderze:")
print(os.getcwd())

Twój Jupyter znajduje się w folderze:
C:\Users\marek


In [6]:
import os
import requests
import pandas as pd
from datetime import datetime
from google.cloud import bigquery

# 1. KONFIGURACJA KLUCZA I PROJEKTU
# Pamiętaj o podaniu poprawnej ścieżki do swojego pliku .json!
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "TUTAJ_WPISZ_NAZWE_PLIKU.json"

PROJECT_ID = "projekt-nbp-499618"
# Używamy nowego zbioru danych z limitem Sandbox (dane_nbp_v2)
TABLE_ID = f"{PROJECT_ID}.dane_nbp_v2.kursy_euro"

# 2. POBRANIE DANYCH Z NBP
url = "https://api.nbp.pl/api/exchangerates/rates/a/eur/?format=json"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    aktualny_kurs = data['rates'][0]['mid']
    
    # 3. PRZYGOTOWANIE WERSJI Z REKORDEM
    nowy_rekord = {
        "data_pobrania": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "kod_waluty": "EUR",
        "kurs": float(aktualny_kurs)
    }

    df = pd.DataFrame([nowy_rekord])
    df['data_pobrania'] = pd.to_datetime(df['data_pobrania'])

    # 4. WYSYŁKA DO BIGQUERY
    client = bigquery.Client(project=PROJECT_ID)
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
    
    job = client.load_table_from_dataframe(df, TABLE_ID, job_config=job_config)
    job.result()  # Czekamy na potwierdzenie z chmury

    print(f" Sukces! Pobrano kurs EUR: {aktualny_kurs} zł i dodano do BigQuery!")
else:
    print(f"Błąd pobierania danych z NBP: {response.status_code}")

C:\Users\marek\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Forbidden: 403 Billing has not been enabled for this project. Enable billing at https://console.cloud.google.com/billing. Table expiration time must be less than 60 days while in sandbox mode.; reason: billingNotEnabled, message: Billing has not been enabled for this project. Enable billing at https://console.cloud.google.com/billing. Table expiration time must be less than 60 days while in sandbox mode.